# KUSI MINIMARKET — Big Data con PySpark (Capítulo 10)
**Curso:** Base de Datos Avanzadas y Big Data (CIIN1021P) · **Entrada:** CSV del Data Warehouse (`dw_listos_para_cargar`) + datos externos · **Salidas:** tabla de tiempos, efecto de eventos y pronóstico base para el Dashboard 2.

Flujo: 1) cargar el DW · 2) estadística descriptiva (DataFrame, SparkSQL y RDD) · 3) datos externos (feriados, festividades, clima) · 4) efecto de eventos · 5) pronóstico base · 6) clima y mix por temporada · 7) benchmark de escalabilidad · 8) exportar para Power BI.

In [ ]:
# --- 0. CONFIGURACIÓN (ajuste las rutas) ---------------------------------------
import os, time, statistics
RUTA_DW   = os.environ.get("KUSI_DW",  r"C:\KUSI_MINIMARKET\dw_listos_para_cargar")   # Colab: "/content/dw"
RUTA_EXT  = os.environ.get("KUSI_EXT", r"C:\KUSI_MINIMARKET\datos_externos")           # carpeta 08_Datos_Externos del ZIP
RUTA_FEST = os.environ.get("KUSI_FEST", os.path.join(RUTA_EXT, "festividades_cajamarca.csv"))
RUTA_CLIMA = os.environ.get("KUSI_CLIMA", os.path.join(RUTA_EXT, "clima_cajamarca_diario.csv"))
SALIDA    = os.environ.get("KUSI_OUT", r"C:\KUSI_MINIMARKET\spark_salidas")
os.makedirs(SALIDA, exist_ok=True)

In [ ]:
# --- 1. SESIÓN SPARK LOCAL -------------------------------------------------------
# Windows: requiere Java 8/11/17 instalado y JAVA_HOME definido.  Colab: !pip install pyspark
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder.master("local[*]").appName("KUSI_BigData")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| núcleos:", spark.sparkContext.defaultParallelism)

## 2. Carga del Data Warehouse (DataFrames)

In [ ]:
def leer(nombre):
    return spark.read.option("header", True).option("inferSchema", True).option("encoding", "UTF-8").csv(os.path.join(RUTA_DW, nombre + ".csv"))
fact, dfecha, dsuc, dprod = leer("FactVentas"), leer("DimFecha"), leer("DimSucursal"), leer("DimProducto")
dfecha = dfecha.withColumn("Fecha", F.to_date("Fecha"))
for n, d in [("FactVentas", fact), ("DimFecha", dfecha), ("DimSucursal", dsuc), ("DimProducto", dprod)]:
    print(f"{n}: {d.count():,} filas")

## 3. Estadística descriptiva con tres objetos Spark: DataFrame, SparkSQL y RDD

In [ ]:
# DataFrame
fact.select("Cantidad", "PrecioUnitario", "VentaNeta").describe().show()
# SparkSQL
fact.createOrReplaceTempView("fact"); dfecha.createOrReplaceTempView("dfecha"); dsuc.createOrReplaceTempView("dsuc")
spark.sql('''
  SELECT s.NombreSucursal, COUNT(DISTINCT f.IdVentaOrigen) AS ventas, ROUND(SUM(f.VentaNeta),2) AS venta_neta,
         ROUND(SUM(f.VentaNeta)/COUNT(DISTINCT f.IdVentaOrigen),2) AS ticket_promedio
  FROM fact f JOIN dsuc s ON s.KeySucursal = f.KeySucursal
  GROUP BY s.NombreSucursal ORDER BY venta_neta DESC''').show(truncate=False)
# RDD: total de venta neta con map/reduce (verificación cruzada)
total_rdd = fact.select("VentaNeta").rdd.map(lambda r: float(r[0])).reduce(lambda a, b: a + b)
total_df  = fact.agg(F.sum("VentaNeta")).first()[0]
print(f"Total VentaNeta  RDD={total_rdd:,.2f}  DataFrame={total_df:,.2f}  diferencia={abs(total_rdd-total_df):.4f}")

## 4. Datos externos
- **Feriados nacionales:** `feriados_peru.csv` (36 fechas; origen *referencial*).
- **Festividades de Cajamarca:** `festividades_cajamarca.csv` (Carnaval, Semana Santa, Corpus Christi, Día de la Madre, etc.). Las de Pascua se **calculan**; las campañas comerciales son **supuestos**. Verificar con el calendario oficial de la Municipalidad Provincial de Cajamarca.
- **Clima diario:** `clima_cajamarca_diario.csv` (temperatura y precipitación; origen *sintético* o *open-meteo* según el archivo). Se usa además la **temporada** (Lluviosa dic–abr · Transición oct–nov · Seca may–sep).
Cada archivo trae la columna `origen`; el informe la declara.

In [ ]:
fer = spark.read.option("header", True).csv(os.path.join(RUTA_EXT, "feriados_peru.csv"))
colnom = next((c for c in fer.columns if c.lower() in ("nombre", "feriado", "descripcion", "descripción")), None)
fer = (fer.withColumn("fecha", F.to_date("fecha")).dropna(subset=["fecha"]).dropDuplicates(["fecha"])
          .select("fecha", F.lit("Feriado nacional").alias("tipo_evento"),
                  (F.col(colnom) if colnom else F.lit("Feriado")).alias("nombre_evento")))
eventos = fer
if os.path.exists(RUTA_FEST) and os.path.getsize(RUTA_FEST) > 0:  # festividades
    fest = spark.read.option("header", True).csv(RUTA_FEST)
    if fest.count() > 0:
        fest = (fest.withColumn("ini", F.to_date("fecha_inicio")).withColumn("fin", F.to_date(F.coalesce("fecha_fin", "fecha_inicio")))
                    .dropna(subset=["ini"])
                    .withColumn("fecha", F.explode(F.sequence("ini", "fin")))
                    .select("fecha", F.lit("Festividad local").alias("tipo_evento"), F.col("nombre").alias("nombre_evento")))
        eventos = eventos.unionByName(fest)
eventos = (eventos.groupBy("fecha").agg(F.min("tipo_evento").alias("tipo_evento"), F.min("nombre_evento").alias("nombre_evento"))
                  .withColumnRenamed("fecha", "fecha_ev"))
print("Fechas con evento:", eventos.count()); eventos.groupBy("tipo_evento").count().show()

## 5. Ventas diarias por sucursal y efecto de los eventos

In [ ]:
diario = (fact.join(dfecha.select("KeyFecha", "Fecha", "DiaSemana", "Anio", "Mes"), "KeyFecha")
              .join(dsuc.filter(F.col("EsOperativa").cast("string").isin("1", "True", "true")).select("KeySucursal", "NombreSucursal"), "KeySucursal")
              .groupBy("KeySucursal", "NombreSucursal", "Fecha", "DiaSemana", "Anio", "Mes")
              .agg(F.round(F.sum("VentaNeta"), 2).alias("venta_neta"), F.countDistinct("IdVentaOrigen").alias("tickets"))
              .join(eventos, F.col("Fecha") == F.col("fecha_ev"), "left").drop("fecha_ev").cache())
diario.count()
# Línea base: promedio de días SIN evento, por sucursal y día de la semana (controla el efecto del día)
base = (diario.filter(F.col("tipo_evento").isNull()).groupBy("KeySucursal", "DiaSemana")
              .agg(F.avg("venta_neta").alias("base_venta")))
efecto = (diario.filter(F.col("tipo_evento").isNotNull()).join(base, ["KeySucursal", "DiaSemana"])
                .withColumn("uplift", F.col("venta_neta") / F.col("base_venta") - 1))
uplift_evento = (efecto.groupBy("NombreSucursal", "tipo_evento")
                       .agg(F.count("*").alias("dias_evento"), F.round(F.avg("uplift") * 100, 1).alias("variacion_pct"))
                       .orderBy("NombreSucursal", "tipo_evento"))
uplift_evento.show(40, truncate=False)

## 6. Pronóstico base y su error (validación con los últimos 28 días)
Se compara un pronóstico por **día de la semana** contra el mismo pronóstico **ajustado por evento**. Se reporta el MAPE de ambos; si el ajuste no mejora, se dice.

In [ ]:
maxf = diario.agg(F.max("Fecha")).first()[0]
corte = F.date_sub(F.lit(maxf), 27)
train = diario.filter(F.col("Fecha") < corte)
test  = diario.filter(F.col("Fecha") >= corte)
base_tr = train.filter(F.col("tipo_evento").isNull()).groupBy("KeySucursal", "DiaSemana").agg(F.avg("venta_neta").alias("base"))
upl_tr = (train.filter(F.col("tipo_evento").isNotNull()).join(base_tr, ["KeySucursal", "DiaSemana"])
               .groupBy("KeySucursal", "tipo_evento").agg(F.avg(F.col("venta_neta") / F.col("base") - 1).alias("upl")))
pred = (test.join(base_tr, ["KeySucursal", "DiaSemana"], "left").join(upl_tr, ["KeySucursal", "tipo_evento"], "left")
            .withColumn("pred_dia", F.col("base"))
            .withColumn("pred_evento", F.col("base") * (1 + F.coalesce("upl", F.lit(0.0)))))
met = pred.filter(F.col("venta_neta") > 0).agg(
    F.round(F.avg(F.abs(F.col("venta_neta") - F.col("pred_dia")) / F.col("venta_neta")) * 100, 2).alias("MAPE_dia_%"),
    F.round(F.avg(F.abs(F.col("venta_neta") - F.col("pred_evento")) / F.col("venta_neta")) * 100, 2).alias("MAPE_con_evento_%"),
    F.count("*").alias("dias_evaluados"))
met.show()
met_row = met.first().asDict()

In [ ]:
# Pronóstico de los 14 días posteriores al último dato, por sucursal (para el Dashboard 2)
base_all = base
fut = (spark.range(1, 15).select(F.date_add(F.lit(maxf), F.col("id").cast("int")).alias("Fecha"))
            .withColumn("DiaSemana", F.dayofweek("Fecha"))
            .join(diario.select("KeySucursal", "NombreSucursal").distinct(), how="cross")
            .join(eventos, F.col("Fecha") == F.col("fecha_ev"), "left").drop("fecha_ev")
            .join(base_all, ["KeySucursal", "DiaSemana"], "left")
            .join(uplift_evento.join(diario.select("KeySucursal", "NombreSucursal").distinct(), "NombreSucursal")
                  .select("KeySucursal", "tipo_evento", (F.col("variacion_pct") / 100).alias("upl")), ["KeySucursal", "tipo_evento"], "left")
            .withColumn("venta_pronosticada", F.round(F.col("base_venta") * (1 + F.coalesce("upl", F.lit(0.0))), 2))
            .select("KeySucursal", "NombreSucursal", "Fecha", "tipo_evento", "nombre_evento", "venta_pronosticada"))
fut.orderBy("NombreSucursal", "Fecha").show(10, truncate=False)

## 6b. Clima y temporada: ¿cambian las ventas y el mix de productos?

In [ ]:
clima = (spark.read.option("header", True).csv(RUTA_CLIMA)
         .select(F.to_date("fecha").alias("fecha_cl"), F.col("precipitacion_mm").cast("double").alias("precip_mm"),
                 F.col("temp_media_c").cast("double").alias("temp_media")))
dcl = (diario.join(clima, F.col("Fecha") == F.col("fecha_cl"), "left").drop("fecha_cl")
             .filter(F.col("tipo_evento").isNull())                      # días sin evento: aísla el efecto del clima
             .join(base, ["KeySucursal", "DiaSemana"]).withColumn("ratio", F.col("venta_neta") / F.col("base_venta"))
             .withColumn("condicion", F.when(F.col("precip_mm") >= 1, "Día lluvioso").otherwise("Día seco")))
efecto_clima = dcl.groupBy("condicion").agg(F.count("*").alias("sucursal_dias"), F.round((F.avg("ratio") - 1) * 100, 2).alias("variacion_pct"))
efecto_clima.show()
corr_temp = dcl.agg(F.round(F.corr("ratio", "temp_media"), 3).alias("corr_ventas_temperatura")).first()[0]
corr_lluvia = dcl.agg(F.round(F.corr("ratio", "precip_mm"), 3).alias("corr_ventas_precipitacion")).first()[0]
print("Correlación (ratio de venta vs temperatura):", corr_temp, "| vs precipitación:", corr_lluvia)

In [ ]:
# Mix de categorías y productos por TEMPORADA (índice 100 = participación anual)
temporada = (F.when(F.col("Mes").isin(12, 1, 2, 3, 4), "Lluviosa").when(F.col("Mes").isin(10, 11), "Transición").otherwise("Seca"))
ft = (fact.join(dfecha.select("KeyFecha", "Mes"), "KeyFecha")
          .join(dprod.select("KeyProducto", "Categoria", "NombreProducto"), "KeyProducto").withColumn("temporada", temporada))
def indice(df, nivel, minimo_ventas=0):
    tot_t = df.groupBy("temporada").agg(F.sum("VentaNeta").alias("tt"))
    tot = df.agg(F.sum("VentaNeta")).first()[0]
    g_t = df.groupBy("temporada", nivel).agg(F.sum("VentaNeta").alias("v")).join(tot_t, "temporada").withColumn("part_t", F.col("v") / F.col("tt"))
    g = df.groupBy(nivel).agg(F.sum("VentaNeta").alias("vt")).filter(F.col("vt") >= minimo_ventas).withColumn("part", F.col("vt") / tot)
    return g_t.join(g, nivel).withColumn("indice", F.round(F.col("part_t") / F.col("part") * 100, 1)).select("temporada", nivel, F.round("v", 2).alias("venta"), "indice")
idx_cat = indice(ft, "Categoria")
top_prod = ft.groupBy("NombreProducto").agg(F.sum("VentaNeta").alias("vt")).orderBy(F.desc("vt")).limit(100).agg(F.min("vt")).first()[0]
idx_prod = indice(ft, "NombreProducto", top_prod)       # solo los 100 productos de mayor venta (evita ruido)
w = Window.partitionBy("temporada").orderBy(F.desc("indice"))
print("Top categorías por temporada (índice > 100 = se vende más de lo habitual):")
idx_cat.withColumn("r", F.row_number().over(w)).filter("r <= 3").orderBy("temporada", "r").show(20, truncate=False)
print("Top productos por temporada:")
idx_prod.withColumn("r", F.row_number().over(w)).filter("r <= 3").orderBy("temporada", "r").show(20, truncate=False)

## 7. Benchmark de escalabilidad (Spark)
Consulta: **facturación por sucursal y mes** (JOIN + GROUP BY). Se replica la tabla de hechos ×1, ×5, ×10 y ×20 y se toma la **mediana de 3 ejecuciones**. La misma consulta se mide en SQL Server con `07_Benchmark_SQLServer.sql`. Los tiempos son los **medidos**; no se comparan con valores de ejemplo.

In [ ]:
def consulta(f):
    return (f.join(dfecha.select("KeyFecha", "Anio", "Mes"), "KeyFecha")
             .groupBy("KeySucursal", "Anio", "Mes").agg(F.sum("VentaNeta").alias("v")).count())
resultados = []
for esc in [1, 5, 10, 20]:
    f = fact if esc == 1 else fact.crossJoin(spark.range(esc).withColumnRenamed("id", "rep")).drop("rep")
    f = f.repartition(8).cache(); filas = f.count()
    consulta(f)                                   # calentamiento (JIT/caché)
    t = []
    for _ in range(3):
        t0 = time.perf_counter(); consulta(f); t.append(time.perf_counter() - t0)
    resultados.append({"escala": esc, "filas": filas, "segundos_mediana": round(statistics.median(t), 3)})
    f.unpersist(); print(resultados[-1])

## 8. Exportación para Power BI (Dashboard 2)

In [ ]:
import pandas as pd
def guardar(df, nombre):
    p = os.path.join(SALIDA, nombre); df.toPandas().to_csv(p, index=False, encoding="utf-8-sig"); print("OK ->", p)
guardar(diario.select("KeySucursal", "NombreSucursal", "Fecha", "DiaSemana", "venta_neta", "tickets", "tipo_evento", "nombre_evento"), "ventas_diarias_sucursal_eventos.csv")
guardar(uplift_evento, "efecto_eventos_por_sucursal.csv")
guardar(fut, "pronostico_14_dias.csv")
guardar(efecto_clima, "efecto_clima.csv")
guardar(idx_cat, "categoria_por_temporada.csv")
guardar(idx_prod, "producto_por_temporada.csv")
pd.DataFrame(resultados).to_csv(os.path.join(SALIDA, "benchmark_spark.csv"), index=False)
pd.DataFrame([met_row]).to_csv(os.path.join(SALIDA, "metricas_pronostico.csv"), index=False)
print(pd.DataFrame(resultados)); print(met_row)